# 01 — eFeature Extraction

Build an `EModelEFeatureExtractionScanConfig`, expand it via `GridScanGenerationTask`,
and run the `EModelEFeatureExtractionTask` for each coordinate. The task calls
[`bluepyefe.extract.extract_efeatures`](https://github.com/BlueBrain/BluePyEfe)
directly on the experimental traces — no `EModel_pipeline`, recipes, morphology
or mechanisms are needed at this stage.

**Reads from:** `obi-output/L_emodel_optimization_inputs/ephys_data/` (created
by `00_setup_download_l5pc_data.ipynb`).

**Writes to:** `obi-output/01_efeature_extraction/grid_scan/0/` — a working
directory containing the copied ephys data, the bluepyefe `extraction/` figures,
and `extracted_features.json` (the serialised `FitnessCalculatorConfiguration`).
The optimisation stage in notebook 02 picks the features file up and slots it
into `config/features/<emodel>.json`.


## Imports

In [1]:
from pathlib import Path

import obi_one as obi
from obi_one.scientific.tasks.emodel_optimization._01_efeature_extraction.blocks import (
    ExtractionInitialize,
)


## Resolve input paths

In [2]:
inputs_root = (
    Path.cwd() / "../../../obi-output/L_emodel_optimization_inputs"
).resolve()
assert inputs_root.exists(), (
    f"{inputs_root} not found — run 00_setup_download_l5pc_data.ipynb first."
)
print("Inputs:", inputs_root)


Inputs: /Users/james/Documents/obi/code/obi-main/obi-output/L_emodel_optimization_inputs


## Build the scan config

Every `bluepyefe` parameter relevant to extraction is exposed via blocks; defaults
match the L5PC recipe (`Threshold=-20`, `interp_step=0.025`, `default_std_value=0.01`, …).
The `targets` block defaults to the same protocols + features as `examples/L5PC/targets.py`.
The only required input is the path to the experimental traces directory.


In [3]:
scan_config = obi.EModelEFeatureExtractionScanConfig(
    initialize=ExtractionInitialize(
        ephys_data_path=inputs_root / "ephys_data" / "C060109A1-SR-C1",
    ),
)
print(scan_config.extraction_settings)


ExtractionSettings(type='ExtractionSettings', plot_extraction=True, default_std_value=0.01, extract_absolute_amplitudes=False, name_rin_protocol='IV_-20', name_rmp_protocol='IV_0')


## Run the grid scan

Per-coordinate output goes to `obi-output/01_efeature_extraction/grid_scan/<idx>/`
(the `ZERO_INDEX` directory option keeps the path stable for downstream notebooks).


In [4]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/01_efeature_extraction/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

obi.run_tasks_for_generated_scan(grid_scan)


[2026-06-05 07:50:08,411] WARNING: Standard deviation for efeatures depol_block_bool stimulus IDrest is 0 and will be set to 0.01


[2026-06-05 07:50:08,412] WARNING: Standard deviation for efeatures mean_frequency stimulus sAHP is 0 and will be set to 0.01


[2026-06-05 07:50:08,413] WARNING: Standard deviation for efeatures voltage_base stimulus sAHP is 0 and will be set to 0.01


[2026-06-05 07:50:08,413] WARNING: Standard deviation for efeatures depol_block_bool stimulus sAHP is 0 and will be set to 0.01


[PosixPath('/Users/james/Documents/obi/code/obi-main/obi-output/01_efeature_extraction/grid_scan/0')]

<Figure size 640x480 with 0 Axes>

## Inspect the extracted features

In [5]:
import json

coord_root = Path(grid_scan.single_configs[0].coordinate_output_root).resolve()
print("Coordinate output:", coord_root)
print()

features_path = coord_root / "extracted_features.json"
print("Features:", features_path)
features = json.loads(features_path.read_text())
print("Top-level keys:", list(features))
print(
    f"Extracted {len(features.get('efeatures', []))} efeatures across"
    f" {len(features.get('protocols', []))} protocols."
)


Coordinate output: /Users/james/Documents/obi/code/obi-main/obi-output/01_efeature_extraction/grid_scan/0

Features: /Users/james/Documents/obi/code/obi-main/obi-output/01_efeature_extraction/grid_scan/0/extracted_features.json
Top-level keys: ['efeatures', 'protocols']
Extracted 42 efeatures across 5 protocols.
